In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
# Download dataset from Kaggle
Q3_data = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(Q3_data)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
print("Missing values before imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])


for col in df.select_dtypes(include=np.number).columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)

print("\nMissing values after imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Task 2:
def check_duplicates(df_param):
  duplicates = df_param.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_param.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
  return df_param

df = check_duplicates(df)

In [ ]:
# Task 3:
categorical_cols = df.select_dtypes(include='object').columns

if len(categorical_cols) > 0:
    print(f"Encoding {len(categorical_cols)} categorical features: {list(categorical_cols)}")
    df = pd.get_dummies(df, columns=categorical_cols, dummy_na=False)
    print("Categorical features encoded.")
else:
    print("No categorical features found for encoding.")

In [ ]:
# Task 4:
numerical_cols_for_scaling = df.select_dtypes(include=np.number).columns.drop('Target', errors='ignore')

if len(numerical_cols_for_scaling) > 0:
    print(f"Applying StandardScaler to {len(numerical_cols_for_scaling)} numerical features.")
    scaler = StandardScaler()
    df[numerical_cols_for_scaling] = scaler.fit_transform(df[numerical_cols_for_scaling])
    print("Numerical features scaled.")
else:
    print("No numerical features found for scaling.")

In [ ]:
# Task 5: Check for target imbalance
target_counts = df['Target'].value_counts(normalize=True)
print("Target variable distribution:")
print(target_counts)

if target_counts.min() < 0.3:
    print("The target variable is imbalanced.")
else:
    print("The target variable is not significantly imbalanced.")

In [ ]:
X = df.drop('Target', axis=1)
y = df['Target']

In [ ]:


from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store F1 scores for each fold
f1_scores = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Initialize and train CatBoostClassifier
    model = CatBoostClassifier(random_state=42, verbose=0, iterations=100)
    model.fit(X_train, y_train)

    # Make predictions on the validation set
    y_pred = model.predict(X_val)

    # Calculate F1 score
    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)
    print(f"Fold {fold+1} F1 Score: {f1:.4f}")

# Print the averaged F1 score across all folds
print(f"\nAverage F1 Score across all folds: {np.mean(f1_scores):.4f}")

In [ ]:
import matplotlib.pyplot as plt


feature_importances = model.get_feature_importance()
feature_names = X.columns


importance_series = pd.Series(feature_importances, index=feature_names)


sorted_importance = importance_series.sort_values(ascending=False)


plt.figure(figsize=(12, 8))
sorted_importance.head(20).plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.xlabel('Feature Importance')
plt.ylabel('Feature Name')
plt.gca().invert_yaxis() # To have the most important feature at the top
plt.show()

In [ ]:

most_important_feature = sorted_importance.index[0]

print(f"The golden feature (most important predictor) is: {most_important_feature}")

In [ ]:
# Task Bonus:
x = df['P_2']
f1_scores_for_x = []
for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[x], X.iloc[val_index]
    y_train, y_val = y.iloc[x], y.iloc[val_index]

    # Initialize and train CatBoostClassifier
    model = CatBoostClassifier(random_state=42, verbose=0, iterations=100)
    model.fit(X_train, y_train)

    # Make predictions on the validation set
    y_pred = model.predict(X_val)

    # Calculate F1 score
    f1 = f1_score(y_val, y_pred)
    f1_scores_for_x.append(f1)
    print(f"Fold {fold+1} F1 Score: {f1:.4f}")

# Print the averaged F1 score across all folds
print(f"\nAverage F1 Score across all folds: {np.mean(f1_scores_for_x):.4f}")